In [ ]:
import numpy as np
%matplotlib ipympl
from matplotlib import pyplot, colors
import ipywidgets

from pygetm import vertical_coordinates, constants, domain

# Generalized vertical coordinates (GVC) and sigma

In [ ]:
D = np.linspace(5.0, 500.0, 50)

fig, (ax_sigma, ax_gvc) = pyplot.subplots(ncols=2, figsize=(10, 5))
for ax in (ax_sigma, ax_gvc):
    ax.set_xlabel("bottom depth (m)")
    ax.set_ylabel("depth (m)")
    ax.set_ylim(D[-1], 0.0)
    ax.set_xlim(0.0, D[-1])
    ax.set_facecolor((0.9, 0.9, 0.9))
artists = []
ax_sigma.set_title("zoomed sigma coordinates")
ax_gvc.set_title("generalized vertical coordinates")

cm = colors.ListedColormap(["white"])


def update(nz=30, ddl=0.5, ddu=0.75, Dgamma=40.0, gamma_surf=True):
    # Remove all exisiting elements from the axes
    while artists:
        artists.pop().remove()

    # Sigma coordinates
    vc_sigma = vertical_coordinates.Sigma(nz, ddl=ddl, ddu=ddu)
    h = vc_sigma(D[np.newaxis, :])[:, 0, :]
    z = np.zeros((nz + 1, h.shape[1]))
    z[1:] = h.cumsum(axis=0)
    dummy = np.zeros((z.shape[0] - 1, z.shape[1] - 1))
    pc = ax_sigma.pcolormesh(D[:], z[-1] - z, dummy, ec="k", lw=0.1, cmap=cm)
    artists.append(pc)

    # Generalized vertical coordinates
    # This may fail for some or all water depths for certain combinations of ddl and ddu
    error = None
    try:
        vc = vertical_coordinates.GVC(nz, ddl=ddl, ddu=ddu, Dgamma=Dgamma, gamma_surf=gamma_surf)
    except Exception as e:
        error = str(e)
    if error is None and vc.D_max > D[0]:
        imax = D.searchsorted(vc.D_max)
        h = vc(D[np.newaxis, :imax])[:, 0, :]
        z = np.zeros((nz + 1, h.shape[1]))
        z[1:] = h.cumsum(axis=0)
        dummy = np.zeros((z.shape[0] - 1, z.shape[1] - 1))
        pc = ax_gvc.pcolormesh(D[:imax], z[-1] - z, dummy, ec="k", lw=0.1, cmap=cm)
        artists.append(pc)
        if imax < D.size:
            artists.append(ax_gvc.axvspan(vc.D_max, D[-1], fc="r"))
    else:
        artists.append(ax_gvc.axvspan(0.0, D[-1], fc="r"))
    if error is not None:
        artists.append(
            ax_gvc.text(
                0.5,
                0.5,
                error,
                ha="center",
                va="center",
                transform=ax.transAxes,
                wrap=True,
            )
        )

    fig.canvas.draw()


ipywidgets.interact(
    update, nz=(2, 100), ddl=(0.0, 3.0), ddu=(0.0, 3.0), Dgamma=(0.01, 100.0)
);

# Adaptive Vertical Coordinates

In [ ]:
ntime = 5000
delta = 20.0
delta_rho = 5.0
delta_t=600.0

dom = domain.create_cartesian(x=[-0.5, 0.5], y=[-0.5, 0.5], interfaces=True, f=0.0, H=100.0)
fig, (ax, ax_cb) = pyplot.subplots(ncols=2, figsize=(10, 5), gridspec_kw={"width_ratios": [20, 1]})

def update(nz=30, ddl=0.0, ddu=0.0, Dgamma=40.0, cNN=0.0, drho=0.):
    ax.cla()
    grid = dom.create_grids(nz, 0, 0)
    grid.ho = grid.array(z=constants.CENTERS)
    grid.hhalf = grid.array(z=constants.CENTERS)
    if ddu == 0.0 and ddl == 0.0:
        Dgamma = 0.0
    if drho == 0.0:
        cNN = 0.0
    c = vertical_coordinates.Adaptive(
        nz=nz,
        ddl=ddl,
        ddu=ddu,
        Dgamma=Dgamma,
        gamma_surf=ddu >= ddl,
        hmin=0.0,
        vfilter=0.0,
        hfilter=0.0,
        cNN=cNN,
        drho=drho,
    )
    NN = grid.array(name="NN", z=constants.INTERFACES)
    grid.array(name="SS", z=constants.INTERFACES)
    c.initialize(grid, logger=dom.logger.getChild("vertical_coordinates"))

    grid.hn[:, 0, 0] = np.random.random_sample(c.nz)
    grid.hn[:, 0, 0] *= grid.D[0, 0] / grid.hn.all_values.sum()
    all_h = np.empty((ntime, c.nz))
    all_nug = np.empty((ntime, c.nz))
    all_NN = np.empty((ntime, c.nz + 1))
    z_if = np.zeros((c.nz + 1,))
    for itime in range(ntime):
        z_if[1:] = grid.hn.all_values[:,0,0].cumsum()
        z_if -= grid.H[0,0]
        NN_norm = 1.0/np.sqrt(2*np.pi)/delta*np.exp(-((z_if + 40) / delta)**2)  # integral of 1
        NN.values[:,0,0] = 9.81/1025*delta_rho*NN_norm
        all_h[itime] = grid.hn.all_values[:,0,0]
        grid.ho.all_values = grid.hn.all_values
        c.update(timestep=delta_t)
        all_nug[itime] = c.nug.all_values[:,0,0]
        all_NN[itime] = NN.values[:,0,0]

    # Buoyancy frequency at layer centers
    NN = 0.5*(all_NN[:,1:]+all_NN[:,:-1])

    # Coordinates of corners for plotting
    z_if = np.zeros((all_h.shape[0]+1, all_h.shape[1]+1))
    hcum = all_h.cumsum(axis=1)
    z_if[1:-1,1:] = 0.5 * (hcum[:-1] + hcum[1:])
    z_if[0] = z_if[1]
    z_if[-1] = z_if[-2]
    z_if -= grid.H[0,0]
    t = delta_t/86400.0*(np.arange(all_h.shape[0] + 1) - 0.5)
    t_2d = np.broadcast_to(t[:, np.newaxis], z_if.shape)

    pc=ax.pcolormesh(t_2d, z_if, NN, lw=0.1)
    cb = fig.colorbar(pc, cax=ax_cb)
    cb.set_label("squared buoyancy frequency (s-2)")
    ax.plot(t, z_if, color='r', lw=.5)
    ax.set_xlabel("time (d)")

ipywidgets.interact(
    update, nz=(2, 100), ddl=(0.0, 3.0), ddu=(0.0, 3.0), Dgamma=(0.01, 100.0), cNN=(0.0, 3.0), drho=(0.0, 1.0)
);

